In [1]:
from google.colab import files
files.upload()

Saving employees.csv to employees.csv
Saving sales.csv to sales.csv


{'employees.csv': b'emp_id,name,department,salary,city\r\n1,Rahul,IT,70000,Hyderabad\r\n2,Sneha,HR,60000,Bangalore\r\n3,Arjun,IT,75000,Chennai\r\n4,Priya,Finance,80000,Hyderabad\r\n5,Karan,IT,50000,Mumbai\r\n6,Amit,HR,58000,Delhi\r\n7,Meera,Finance,82000,Bangalore\r\n8,Ravi,IT,72000,Hyderabad\r\n9,Neha,HR,61000,Chennai\r\n10,Vikram,Finance,90000,Delhi\r\n11,Anita,IT,65000,Bangalore\r\n12,Manoj,HR,62000,Mumbai\r\n13,Divya,IT,77000,Hyderabad\r\n14,Sanjay,Finance,85000,Chennai\r\n15,Pooja,IT,69000,Bangalore\r\n16,Kunal,HR,61000,Delhi\r\n17,Sonal,Finance,88000,Mumbai\r\n18,Deepak,IT,73000,Hyderabad\r\n19,Ritu,HR,60000,Chennai\r\n20,Akash,Finance,91000,Delhi',
 'sales.csv': b'sale_id,emp_id,product,quantity,price\r\n1,1,Laptop,1,75000\r\n2,2,Mouse,3,500\r\n3,3,Keyboard,2,1500\r\n4,1,Monitor,1,12000\r\n5,4,Laptop,1,75000\r\n6,3,Mouse,2,500\r\n7,5,Keyboard,1,1500\r\n8,1,Laptop,1,75000\r\n9,6,Mouse,4,500\r\n10,7,Monitor,2,12000\r\n11,8,Keyboard,2,1500\r\n12,9,Mouse,3,500\r\n13,10,Laptop,1,7500

In [4]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('orders').getOrCreate()

In [5]:
df_emp = spark.read.csv("employees.csv", header=True, inferSchema=True)
df_sales = spark.read.csv("sales.csv", header=True, inferSchema=True)

In [6]:
from pyspark.sql.functions import col

df_sales = df_sales.withColumn("revenue", col("quantity") * col("price"))

In [7]:
df_joined = df_sales.join(df_emp, on="emp_id", how="inner")
df_joined.show()

+------+-------+--------+--------+-----+-------+------+----------+------+---------+
|emp_id|sale_id| product|quantity|price|revenue|  name|department|salary|     city|
+------+-------+--------+--------+-----+-------+------+----------+------+---------+
|     1|      8|  Laptop|       1|75000|  75000| Rahul|        IT| 70000|Hyderabad|
|     1|      4| Monitor|       1|12000|  12000| Rahul|        IT| 70000|Hyderabad|
|     1|      1|  Laptop|       1|75000|  75000| Rahul|        IT| 70000|Hyderabad|
|     2|      2|   Mouse|       3|  500|   1500| Sneha|        HR| 60000|Bangalore|
|     3|      6|   Mouse|       2|  500|   1000| Arjun|        IT| 75000|  Chennai|
|     3|      3|Keyboard|       2| 1500|   3000| Arjun|        IT| 75000|  Chennai|
|     4|      5|  Laptop|       1|75000|  75000| Priya|   Finance| 80000|Hyderabad|
|     5|      7|Keyboard|       1| 1500|   1500| Karan|        IT| 50000|   Mumbai|
|     6|      9|   Mouse|       4|  500|   2000|  Amit|        HR| 58000|   

In [8]:
from pyspark.sql.functions import sum, col

emp_revenue = df_joined.groupBy("name") \
    .agg(sum("revenue").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

emp_revenue.show()

+------+-------------+
|  name|total_revenue|
+------+-------------+
| Rahul|       162000|
| Sonal|        75000|
| Divya|        75000|
| Priya|        75000|
|Vikram|        75000|
| Meera|        24000|
| Anita|        12000|
| Pooja|        12000|
| Arjun|         4000|
|  Ravi|         3000|
| Manoj|         3000|
|  Amit|         2000|
| Kunal|         1500|
|Sanjay|         1500|
| Sneha|         1500|
|  Neha|         1500|
| Karan|         1500|
+------+-------------+



In [9]:
emp_revenue.show(5)

+------+-------------+
|  name|total_revenue|
+------+-------------+
| Rahul|       162000|
| Priya|        75000|
| Divya|        75000|
| Sonal|        75000|
|Vikram|        75000|
+------+-------------+
only showing top 5 rows


In [10]:
dept_revenue = df_joined.groupBy("department") \
    .agg(sum("revenue").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

dept_revenue.show()

+----------+-------------+
|department|total_revenue|
+----------+-------------+
|        IT|       269500|
|   Finance|       250500|
|        HR|         9500|
+----------+-------------+



In [11]:
emp_revenue.write.csv("final_sales_report.csv", header=True)